# 04 - XLM-R fine-tuning (Phases 8 and 9) - run this notebook in GOOGLE COLAB

**Before you start (once)**
1. Colab: **Runtime > Change runtime type > T4 GPU > Save**.
2. Google Drive folder `MyDrive/hate-speech/data/` must contain: `train.csv`, `val.csv`, `test.csv`, `zero_shot_test.csv`, `indicguard_external.csv`.
3. Have your GitHub token ready (read-only, for this repository).
4. In the bootstrap cell below, replace `YOUR-GITHUB-USERNAME` with your GitHub user name.

Run the cells **top to bottom**, one at a time. Cell order: bootstrap -> GPU check -> data check -> token length -> **smoke test (2 minutes)** -> full training -> curves -> export.

## 1. Bootstrap: Drive, code from GitHub, libraries, data

In [ ]:
# ===================== Colab bootstrap (identical cell in notebooks 04 and 05) =====================
import os, sys, subprocess, getpass, shutil

GITHUB_USER = "YOUR-GITHUB-USERNAME"                 # <-- EDIT: your GitHub user name
REPO_NAME   = "multilingual-hate-speech"
DRIVE_ROOT  = "/content/drive/MyDrive/hate-speech"   # the folder you created in Google Drive

from google.colab import drive
drive.mount("/content/drive")

REPO_DIR = f"/content/{REPO_NAME}"
if not os.path.isdir(REPO_DIR):
    token = getpass.getpass("GitHub token (what you type is hidden): ")
    url = f"https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git"
    r = subprocess.run(["git", "clone", url, REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("git clone failed:\n" + r.stderr.replace(token, "***"))
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin",
                    f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"], check=True)   # do not keep the token
    print("cloned", REPO_NAME)
else:
    print("repo already cloned (to get newer code: Runtime > Disconnect and delete runtime, then run again)")
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)

# copy the data files from Google Drive into the repo folders
FILES = {"train.csv": "data/processed", "val.csv": "data/processed", "test.csv": "data/processed",
         "zero_shot_test.csv": "data/processed", "indicguard_external.csv": "data/external_test"}
for name, dest in FILES.items():
    src = f"{DRIVE_ROOT}/data/{name}"
    if os.path.exists(src):
        os.makedirs(dest, exist_ok=True); shutil.copy(src, f"{dest}/{name}"); print("copied", name)
    else:
        print("!! MISSING in Google Drive:", src)

## 2. GPU check

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU found! Runtime > Change runtime type > T4 GPU > Save, then run again from the top."
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

## 3. Data check

In [ ]:
import pandas as pd
from src.data.io_utils import read_processed_csv
tr, va = read_processed_csv("data/processed/train.csv"), read_processed_csv("data/processed/val.csv")
print(f"train {len(tr):,} | val {len(va):,}")
print("columns:", list(tr.columns))
print("train hate share: %.1f%%" % (100 * tr["label"].mean()))
tr[["text", "label", "language", "source"]].sample(3, random_state=1)

## 4. Token length: is `max_length = 128` enough?
The plan starts with 128 unless long texts are common. The cell prints how many comments would be cut off.

In [ ]:
import numpy as np
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("xlm-roberta-base")
s = tr.sample(10000, random_state=42)
n_tok = np.array([len(x) for x in tok(s["text"].fillna("").tolist(), add_special_tokens=True)["input_ids"]])
print("tokens per comment: median %d | 95th pct %d | 99th pct %d | max %d" % (np.median(n_tok), np.percentile(n_tok, 95), np.percentile(n_tok, 99), n_tok.max()))
print("share of comments longer than 128 tokens (would be truncated): %.2f%%" % (100 * (n_tok > 128).mean()))
print("-> keep max_length=128" if (n_tok > 128).mean() < 0.05 else "-> consider max_length=192 (more than 5% would be cut)")

## 5. Smoke test (2-3 minutes)
Trains on only 2,000 comments for 1 epoch. **Purpose: find setup problems cheaply** before the real run.
If this cell fails, copy the red error text and send it to me - do not start the full training.

In [ ]:
from src.model.train import TrainConfig, train
smoke = train(TrainConfig(max_train_samples=2000, max_val_samples=1000, epochs=1, output_dir="/content/smoke_model"))
print("SMOKE TEST PASSED - macro-F1 after 2,000 samples:", round(smoke["best_val_macro_f1"], 3), "(any number above 0.5 is fine here)")

## 6. Full training (Phase 9)
Settings: XLM-RoBERTa-base, max_length 128, learning rate 2e-5, batch size 32, 3 epochs, class-weighted loss.
Expect roughly **15-30 minutes per epoch** on a free T4 (an estimate - it varies). **Keep this browser tab open.**
After every epoch the validation loss, macro-F1, precision and recall are printed; the best epoch (by macro-F1) is saved to Google Drive.

If you get "CUDA out of memory": Runtime > Restart session, run the bootstrap again, and use `TrainConfig(batch_size=16, output_dir=OUT_DIR)`.

In [ ]:
OUT_DIR = f"{DRIVE_ROOT}/xlm_roberta_hate"          # saved on Google Drive so it survives a disconnect
result = train(TrainConfig(output_dir=OUT_DIR))

## 7. Learning curves

In [ ]:
import pandas as pd
from matplotlib.figure import Figure
from IPython.display import display, Image
hist = pd.DataFrame(result["history"])
display(hist[["epoch", "train_loss", "val_loss", "macro_f1", "precision_hate", "recall_hate", "minutes_elapsed"]].round(4))
fig = Figure(figsize=(10, 3.8)); a1, a2 = fig.subplots(1, 2)
a1.plot(hist["epoch"], hist["train_loss"], "o-", label="train loss"); a1.plot(hist["epoch"], hist["val_loss"], "o-", label="validation loss")
a1.set_xlabel("epoch"); a1.set_title("Loss"); a1.legend()
a2.plot(hist["epoch"], hist["macro_f1"], "o-", color="green"); a2.set_xlabel("epoch"); a2.set_title("Validation macro-F1")
for a in (a1, a2): a.set_xticks(hist["epoch"])
fig.tight_layout(); os.makedirs("results/figures", exist_ok=True); fig.savefig("results/figures/xlmr_training_curves.png", dpi=150)
display(Image("results/figures/xlmr_training_curves.png"))
print("Best epoch:", result["best_epoch"], "| best validation macro-F1: %.4f" % result["best_val_macro_f1"])

## 8. Export
Copies the training history and the curves to Google Drive (`results_export/`) and lists the saved model files.
**Then download the model folder** to your computer - see the instructions in the chat.

In [ ]:
os.makedirs(f"{DRIVE_ROOT}/results_export", exist_ok=True)
for p in ["results/metrics/xlmr_training_history.json", "results/figures/xlmr_training_curves.png"]:
    shutil.copy(p, f"{DRIVE_ROOT}/results_export/")
print("Model files in Google Drive:")
for fn in sorted(os.listdir(OUT_DIR)):
    print(f"  {fn:<28}{os.path.getsize(os.path.join(OUT_DIR, fn)) / 1e6:>10.1f} MB")
print("\nNext: notebook 05 (evaluation).")